In [21]:
# Importing setup file
from setup import *

# Link directory to save figures
out_path = "output/p1"
os.makedirs(out_path, exist_ok=True)

### Processing text

In [22]:
# loading the text
text = load_file(False, filePath=BASE_DIR / "data/text.txt")
chars, chars_to_idx, idx_to_chars, encoded_text = vocab(translation=False, dataset=text)

print(f'Using device: {device}')

Using device: cuda


### Hyperparameters

In [23]:

nhead = 2
epochs = 100
num_layers = 2
batch_size = 128
seq_len = [10, 20, 30]
input_size = len(chars)
output_size = input_size

## Helper Functions

In [24]:
def train_model(model, train_loader, val_loader):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Initializing variables
    train_losses = []
    val_losses = []
    val_accs = []

    start_time = time.time()

    for epoch in range(epochs):
        # Training loop
        model.train()
        
        for train_x, train_y in train_loader:
            X_train = train_x.to(device)
            Y_train = train_y.to(device)

            optimizer.zero_grad()
            train_output = model(X_train)
            train_loss = criterion(train_output.transpose(1, 2), Y_train)
            train_loss.backward()
            optimizer.step()

            train_losses.append(train_loss.item())
        
        # Validation loop
        model.eval()
        with torch.no_grad():
            for val_x, val_y in val_loader:
                X_val = val_x.to(device)
                Y_val = val_y.to(device)

                val_output = model(X_val)
                val_loss = criterion(val_output.transpose(1, 2), Y_val)
                val_losses.append(val_loss.item())

                _, predicted = torch.max(val_output, 2)
                val_acc = (predicted == Y_val).float().mean()
                val_accs.append(val_acc.item())

        if (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1}:\n Loss: {train_loss.item():.3f}, Validation Loss: {val_loss.item():.3f}, Validation Accuracy: {val_acc.item():.4f}')
    
    end_time = time.time()
    training_time = compute_time(start_time, end_time)

    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_acc': val_accs,
        'final_train_loss': train_losses[-1],
        'final_val_loss': val_losses[-1],
        'final_val_acc': val_accs[-1],
        'train_time': training_time,
    }

## Main Function

In [25]:
model = charTransformer(input_size, hidden_size, output_size, num_layers, nhead).to(device)
results = {}

for seq in seq_len:
    dataset = CharDataset(encoded_text, seq_len=seq)
    train_loader, val_loader = build_loaders(dataset=dataset, batch_size=batch_size)
    results[seq] = {}
    print("\n")
    print("="*50)
    print(f'Sequence length: {seq}')
    print("="*50)

    metrics = train_model(model, train_loader, val_loader)
    model_size = compute_model_size(model)

    
    sample_input = torch.randint(0, input_size, (1, seq)).to(device)
    flops, params = compute_flops(model, sample_input)

    results[seq] = {
        **metrics,
        'model_size_mb': model_size,
        'complexity': flops,
        'no_params': params,
    }

for metric_name, key, fmt in [
    ("Final Train Loss",    "final_train_loss", ".4f"),
    ("Final Val Loss",      "final_val_loss",   ".4f"),
    ("Val Accuracy",        "final_val_acc",    ".4f"),
    ("FLOPs",               "complexity",       ".2f"),
    ("Train Time (s)",      "train_time",       ".2f"),
    ("No. of Params",       "no_params",           ".3f"),
    ("Model Size (MB)",     "model_size_mb",          ".4f"),
]:
    print("\n")
    print(f'{metric_name}')
    for seq in seq_len:
       print(f'Sequence Length: {seq}\t Value: {results[seq][key]}')



Sequence length: 10
Epoch 10:
 Loss: 0.231, Validation Loss: 0.223, Validation Accuracy: 0.9337
Epoch 20:
 Loss: 0.135, Validation Loss: 0.208, Validation Accuracy: 0.9424
Epoch 30:
 Loss: 0.133, Validation Loss: 0.229, Validation Accuracy: 0.9413
Epoch 40:
 Loss: 0.091, Validation Loss: 0.254, Validation Accuracy: 0.9424
Epoch 50:
 Loss: 0.038, Validation Loss: 0.273, Validation Accuracy: 0.9467
Epoch 60:
 Loss: 0.058, Validation Loss: 0.309, Validation Accuracy: 0.9446
Epoch 70:
 Loss: 0.033, Validation Loss: 0.290, Validation Accuracy: 0.9478
Epoch 80:
 Loss: 0.048, Validation Loss: 0.309, Validation Accuracy: 0.9467
Epoch 90:
 Loss: 0.046, Validation Loss: 0.310, Validation Accuracy: 0.9457
Epoch 100:
 Loss: 0.025, Validation Loss: 0.309, Validation Accuracy: 0.9457


Sequence length: 20
Epoch 10:
 Loss: 0.049, Validation Loss: 0.081, Validation Accuracy: 0.9761
Epoch 20:
 Loss: 0.035, Validation Loss: 0.095, Validation Accuracy: 0.9767
Epoch 30:
 Loss: 0.019, Validation Loss: 0.